# Task 4: VAE — Load Weights & Show Per-Class Reconstructions

This notebook loads the trained `ConvVAE` weights saved by `Task4_VAE_v2.ipynb`
(`../weights/vae_mnist_conv_latent2.pt`) and displays one reconstruction per
MNIST digit class (0–9), in the same original-vs-reconstruction grid format
used in the training notebook's `show_reconstructions` function.

## 0. Setup

In [ ]:
import torch
import torch.nn as nn
import torchvision
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_DIR = "../data"
WEIGHTS_DIR = "../weights"
WEIGHTS_PATH = os.path.join(WEIGHTS_DIR, "vae_mnist_conv_latent2.pt")

print("Using device:", device)

# Global plotting style -- matches the training notebook
sns.set_theme(style="whitegrid", context="notebook", palette="deep")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.titleweight"] = "bold"


## 1. Model Definition

Same `ConvVAE` architecture as in `Task4_VAE_v2.ipynb`, needed to load the saved `state_dict`.

In [ ]:
class ConvVAE(nn.Module):
    """3-layer convolutional VAE for 28x28 MNIST digits.

    Encoder: Conv2d x3 (stride-2 downsampling, 28 -> 14 -> 7 -> 4) -> FC(mu), FC(logvar)
    Decoder: FC -> ConvTranspose2d x3 (mirrored upsampling, 4 -> 7 -> 14 -> 28), outputs logits
    """

    def __init__(self, latent_dim=2):
        super().__init__()
        self.latent_dim = latent_dim

        # --- Encoder q_phi(z | x): 3 conv layers ---
        self.enc = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, stride=2, padding=1),    # 28x28 -> 14x14
            nn.BatchNorm2d(32), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),   # 14x14 -> 7x7
            nn.BatchNorm2d(64), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),  # 7x7 -> 4x4
            nn.BatchNorm2d(128), nn.LeakyReLU(0.2, inplace=True),
        )
        self.enc_feat_dim = 128 * 4 * 4
        self.enc_mu = nn.Linear(self.enc_feat_dim, latent_dim)
        self.enc_logvar = nn.Linear(self.enc_feat_dim, latent_dim)

        # --- Decoder p_theta(x | z): FC projection + 3 mirrored transposed-conv layers ---
        self.dec_fc = nn.Linear(latent_dim, self.enc_feat_dim)
        self.dec = nn.Sequential(
            nn.ConvTranspose2d(128, 64, kernel_size=3, stride=2, padding=1, output_padding=0),  # 4x4 -> 7x7
            nn.BatchNorm2d(64), nn.LeakyReLU(0.2, inplace=True),
            nn.ConvTranspose2d(64, 32, kernel_size=3, stride=2, padding=1, output_padding=1),   # 7x7 -> 14x14
            nn.BatchNorm2d(32), nn.LeakyReLU(0.2, inplace=True),
            nn.ConvTranspose2d(32, 1, kernel_size=3, stride=2, padding=1, output_padding=1),    # 14x14 -> 28x28
        )  # final layer outputs logits (sigmoid applied via BCEWithLogits, not inside the module)

    def encode(self, x):
        h = self.enc(x).flatten(1)
        return self.enc_mu(h), self.enc_logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + std * eps

    def decode(self, z):
        h = self.dec_fc(z).view(-1, 128, 4, 4)
        return self.dec(h)  # logits, shape (B, 1, 28, 28)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        logits = self.decode(z)
        return logits, mu, logvar


## 2. Load Test Data

In [ ]:
tfms = transforms.Compose([transforms.ToTensor()])  # already in [0, 1], no extra normalization

test_set = datasets.MNIST(root=DATA_DIR, train=False, download=True, transform=tfms)

print(f"Test set: {len(test_set)} images")


## 3. Load Trained Weights

In [ ]:
LATENT_DIM = 2
vae = ConvVAE(latent_dim=LATENT_DIM).to(device)
vae.load_state_dict(torch.load(WEIGHTS_PATH, map_location=device))
vae.eval()

n_params = sum(p.numel() for p in vae.parameters())
print(f"Loaded weights from: {WEIGHTS_PATH}")
print(f"VAE parameters: {n_params:,}")


## 4. Reconstructions — One Image per Class

Same visualization format as `show_reconstructions` in the training notebook
(original row on top, reconstruction row on the bottom), but with exactly one
example picked from each of the 10 MNIST digit classes (0-9) instead of a
random sample.

In [ ]:
@torch.no_grad()
def show_reconstructions_by_class(model, dataset, n_classes=10, seed=None):
    model.eval()

    # find one dataset index per class label
    rng = np.random.default_rng(seed)
    targets = np.array(dataset.targets)
    idx = []
    for c in range(n_classes):
        class_idx = np.where(targets == c)[0]
        idx.append(rng.choice(class_idx))

    imgs = torch.stack([dataset[i][0] for i in idx]).to(device)

    logits, mu, logvar = model(imgs)
    recon = torch.sigmoid(logits).cpu()

    n = len(idx)
    fig, axes = plt.subplots(2, n, figsize=(1.4 * n, 3.2))
    for i in range(n):
        axes[0, i].imshow(imgs[i, 0].cpu(), cmap="gray"); axes[0, i].axis("off")
        axes[1, i].imshow(recon[i, 0], cmap="gray"); axes[1, i].axis("off")
        axes[0, i].set_title(str(i), fontsize=9)

    # row labels (kept visible even though axis("off") hides ticks/spines -- text artists still render)
    axes[0, 0].text(-0.35, 0.5, "Original", transform=axes[0, 0].transAxes,
                     rotation=90, va="center", ha="center", fontsize=10, fontweight="bold")
    axes[1, 0].text(-0.35, 0.5, "Reconstruction", transform=axes[1, 0].transAxes,
                     rotation=90, va="center", ha="center", fontsize=10, fontweight="bold")

    fig.suptitle("VAE reconstructions: original (top) vs. decoded (bottom) -- one per class", fontweight="bold")
    plt.tight_layout(); plt.show()

show_reconstructions_by_class(vae, test_set, n_classes=10, seed=42)


## 5. Generation of New Samples

20 samples drawn from the prior $z \sim \mathcal{N}(0, I)$ and decoded, shown as two rows of 10, with the sampled latent coordinate above each digit.

In [ ]:
@torch.no_grad()
def generate_samples(model, n=10, latent_dim=2):
    model.eval()
    z = torch.randn(n, latent_dim).to(device)
    logits = model.decode(z)
    samples = torch.sigmoid(logits).cpu()
    return samples, z.cpu()

samples, z_samples = generate_samples(vae, n=20, latent_dim=LATENT_DIM)

fig, axes = plt.subplots(2, 10, figsize=(15, 3.6), gridspec_kw={"hspace": 0.05, "wspace": 0.1})
for i in range(10):
    axes[0, i].imshow(samples[i, 0], cmap="gray")
    axes[0, i].axis("off")
    # per-sample key: the 2D prior coordinate z that was decoded into this digit
    axes[0, i].set_title(f"z=({z_samples[i, 0]:.1f}, {z_samples[i, 1]:.1f})", fontsize=7, pad=2)

    axes[1, i].imshow(samples[10 + i, 0], cmap="gray")
    axes[1, i].axis("off")
    # per-sample key: the 2D prior coordinate z that was decoded into this digit
    axes[1, i].set_title(f"z=({z_samples[10 + i, 0]:.1f}, {z_samples[10 + i, 1]:.1f})", fontsize=7, pad=2)

fig.suptitle("Samples generated from z ~ N(0, I), decoded by the VAE", fontweight="bold", y=0.98)
plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.show()
